# Multi-Model Interactive Chat System

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://sdmntprwestus.oaiusercontent.com/files/00000000-8fac-6230-ac97-0af7a3c73840/raw?se=2025-04-10T18%3A10%3A19Z&sp=r&sv=2024-08-04&sr=b&scid=67ee5f00-affb-577c-a41e-87dfe878a6d8&skoid=72d71449-cf2f-4f10-a498-f160460104ee&sktid=a48cca56-e6da-484e-a814-9c849652bcb3&skt=2025-04-10T10%3A04%3A37Z&ske=2025-04-11T10%3A04%3A37Z&sks=b&skv=2024-08-04&sig=Qfb2s7yyoMN/WXIO7wHDZB3x4f7lcB4RgI1TUsM472w%3D"> 
</p>
</div>

## Description:

This system facilitates dynamic, multi-turn conversations between users and AI models (Anthropic & OpenAI). It alternates between two models to generate responses, ensuring comprehensive and diverse insights.

- Manages the conversation flow between the user and multiple AI models.

- Provides real-time interaction through adaptive response temperature for each prompt.

- Tracks and displays the entire conversation history for transparency.

- Offers flexibility with conversation length and model response customization.

- Implements robust error handling to ensure reliable performance and response quality.





## Step 1: Environment Setup and Installation

This step installs dependencies from `requirements.txt` and checks for `OPENAI_API_KEY`.  

If installation fails, it retries up to 3 times before exiting.  

Once complete, it clears the output and prints a success message.  


In [ ]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output
import os

requirements_installed = False
max_retries = 3
retries = 0



def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    install_status = os.system("pip install -r requirements.txt")
    if install_status == 0:
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return



install_requirements()
clear_output()

print("🚀 Setup complete. Continue to the next cell.")

## Step 2: Environment Variable Setup

This step loads environment variables from `.env` using `dotenv`.  

It checks if `OPENAI_API_KEY` is set; if missing, it exits.  

After validation, it confirms successful setup.  


In [ ]:
from dotenv import load_dotenv

REQUIRED_ENV_VARS = []



def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True)

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)

setup_env()

## Step 3: Adaptive Completion System with Temperature Control

- This script sets up a system to query LLMs (Anthropic and OpenAI) with dynamic temperature control using LiteLLM and Instructor.

- It defines a Pydantic model to validate temperature values and adapt the creativity of model responses based on the prompt.

- The `adaptive_temperature` function calculates the optimal temperature by prompting a language model to suggest a suitable value.

- Two helper functions (`get_completion_anthropic` and `get_completion_openai`) are used to get completions from the respective APIs with the dynamically chosen temperature.

- The implementation includes robust error handling to fall back to a default temperature if any step fails.


In [ ]:
from litellm import completion
import instructor
import traceback
from pydantic import BaseModel

client = instructor.from_litellm(completion=completion)

class AdaptiveTemperature(BaseModel):
    """
    A class to handle the adaptive temperature for the model.
    """
    temperature: float


DEFAULT_FALLBACK_TEMPERATURE = 0.5
ADAPTIVE_TEMPERATURE_MODEL = "anthropic/claude-3-7-sonnet-latest"
DEFAULT_ANTHROPIC_MODEL = "anthropic/claude-3-7-sonnet-latest"
DEFAULT_OPENAI_MODEL = "openai/gpt-4o"

def adaptive_temperature(prompt: str) -> float:
    """
    Applies adaptive temperature to the prompt.
    Args:
        prompt (str): The prompt to send to the API.
    Returns:
        float: The adaptive temperature for the prompt.
    """
    temperature = DEFAULT_FALLBACK_TEMPERATURE
    try:
        user_prompt = f"""
            Based on the prompt below, please provide a temperature value between 0 and 1 that would be appropriate for the given prompt.

            Respond with a value between 0 to 1 only.

            Prompt: {prompt}
        """
        response = client.chat.completions.create(
            model=ADAPTIVE_TEMPERATURE_MODEL,
            messages=[
                {"role": "user", "content": user_prompt}
            ],
            response_model=AdaptiveTemperature,
        )
        temperature = response.temperature
        if temperature < 0 or temperature > 1:
            raise ValueError(f"Temperature out of bounds: {temperature}")
        print(f"Adaptive temperature computed: {temperature}")
        return temperature
    except Exception as e:
        print(f"Failed to compute adaptive temperature, resorting to fallback temperature[{temperature}]: {e}")
        traceback.print_exc()
        return temperature

def get_completion_anthropic(prompt: str) -> str:
    """
    Gets the completion from the Anthropic API.
    Args:
        prompt (str): The prompt to send to the API.
    Returns:
        str: The completion from the API.
    """
    try:
        response = completion(
            model=DEFAULT_ANTHROPIC_MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=adaptive_temperature(prompt),
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Failed to get completion from Anthropic API: {e}")
        traceback.print_exc()
        return "Failed to get completion from Anthropic API."
    
def get_completion_openai(prompt: str) -> str:
    """
    Gets the completion from the OpenAI API.
    Args:
        prompt (str): The prompt to send to the API.
    Returns:
        str: The completion from the API.
    """
    try:
        response = completion(
            model=DEFAULT_OPENAI_MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=adaptive_temperature(prompt),
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Failed to get completion from OpenAI API: {e}")
        traceback.print_exc()
        return "Failed to get completion from OpenAI API."

## Step 4: Triggering Adaptive Temperature Calculation

- The `adaptive_temperature` function is called with the prompt "What is the capital of France?" to compute an optimal temperature for the given input.

- It determines how creative or precise the model's response should be, based on the context of the prompt.


In [ ]:
adaptive_temperature("What is the capital of France?")

## Step 5: Displaying Model Responses for a Given Prompt

- The code sets a prompt, "What is life?", and queries both the Anthropic and OpenAI APIs for their respective responses.

- It prints the responses from each model to the console and then displays them as formatted Markdown for better readability.

- The `get_completion_anthropic` and `get_completion_openai` functions fetch the responses based on the dynamically calculated temperature for each model.


In [ ]:
from IPython.display import display, Markdown

prompt = "What is life?"

print("Anthropic API response:")
anthropic_response = get_completion_anthropic(prompt)
display(Markdown(f"**Anthropic API response:**\n\n{anthropic_response}"))
print("\n\n")
print("OpenAI API response:")
openai_response = get_completion_openai(prompt)
display(Markdown(f"**OpenAI API response:**\n\n{openai_response}"))

## Step 6: Managing Chat History

- The `ChatHistory` class is designed to store and manage a conversation's history.

- It allows adding messages with the sender's ID and content using the `add_message` method, which appends the message to the history list.

- The `get_history` method retrieves the last `n` messages (default is 10) from the history, ensuring the conversation doesn't exceed the specified window.

- The `length` method returns the current number of messages in the chat history.


In [ ]:

DEFAULT_HISTORY_WINDOW_LENGTH = 10

class ChatHistory:
    """
    A class to manage chat history.
    """

    def __init__(self):
        self.history = []

    def add_message(self, sender_id: str, content: str):
        """
        Adds a message to the chat history.
        Args:
            role (str): The role of the message sender (user or assistant).
            content (str): The content of the message.
        """
        self.history.append(f"{sender_id}: {content}")

    def get_history(self, n = DEFAULT_HISTORY_WINDOW_LENGTH) -> list:
        """
        Gets the chat history.
        Returns:
            list: The chat history.
        """
        return self.history[-n:]
    
    def length(self) -> int:
        """
        Gets the length of the chat history.
        Returns:
            int: The length of the chat history.
        """
        return len(self.history)


## Step 7: Managing Conversation Flow

- The `Counsel` class handles interactions with both the Anthropic and OpenAI models by managing the conversation history.

- The `start` method begins the conversation by sending the initial prompt from the user and then alternating between the models (Anthropic and OpenAI) for responses.

- Each model’s response is appended to the chat history until the maximum conversation length is reached (default is 5).

- The `display_history` method prints the entire chat history, showing the sequence of user and model messages.


In [ ]:

DEFAULT_MAX_CONVERSATION_LENGTH = 5

class Counsel:
    """
    A class to manage the conversation with the model.
    """

    def __init__(self, model: str = DEFAULT_ANTHROPIC_MODEL):
        self.model = model
        self.history = ChatHistory()

    def start(self, prompt: str, max_conversation_length: int = DEFAULT_MAX_CONVERSATION_LENGTH):
        """
        Starts the conversation with the model.
        Args:
            prompt (str): The prompt to send to the model.
        """
        message_count = 0

        self.history.add_message("User", prompt)
        message_count += 1

        while message_count < max_conversation_length:
            # Get Anthropic response
            anthropic_prompt = f"You are a member of a chat counsel, respond to the last message in the chat representing yourself, Anthropic. \n\nHistory: {self.history.get_history()}"
            anthropic_response = get_completion_anthropic(anthropic_prompt)
            self.history.add_message("Anthropic", anthropic_response)
            message_count += 1
            openai_prompt = f"You are a member of a chat counsel, respond to the last message in the chat representing yourself, Open AI. Just respond with your message, don't respond like this 'OpenAI: my response' just type your response. \n\nHistory: {self.history.get_history()}"
            openai_response = get_completion_openai(openai_prompt)
            self.history.add_message("OpenAI", openai_response)
            message_count += 1

            if message_count >= max_conversation_length:
                break

    
    def display_history(self):
        """
        Displays the chat history.
        """
        for message in self.history.get_history(n=self.history.length()):
            print(message)
        print("\n\n")

## Step 8: Starting a Conversation and Displaying History

- The code sets the prompt "Are humans more intelligent than AI?" and initializes a `Counsel` object.

- It starts the conversation with the given prompt by calling the `start` method, which generates responses from both the Anthropic and OpenAI models.

- Finally, it displays the conversation history using the `display_history` method, showing the interaction between the user and the models.


In [ ]:
prompt = "Are humans more intelligent than AI?"

counsel = Counsel()
counsel.start(prompt)
counsel.display_history()

## Conclusion

- The code implements an interactive system that facilitates conversation between a user and two AI models (Anthropic and OpenAI).

- It dynamically adjusts the model's response temperature based on the prompt to control creativity or precision.

- The `ChatHistory` class manages the conversation context, ensuring that previous messages are considered when generating responses.

- The `Counsel` class handles the flow of the conversation, alternating between the two models, and displays the chat history for review.

- This setup allows for an ongoing, multi-turn dialogue where both models participate in the conversation, responding to each other in a structured manner.


---

# Thank You for visiting The Hackers Playbook! 🌐

If you liked this research material;

- [Subscribe to our newsletter.](https://thehackersplaybook.substack.com)

- [Follow us on LinkedIn.](https://www.linkedin.com/company/the-hackers-playbook/)

- [Leave a star on our GitHub.](https://www.github.com/thehackersplaybook)

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
</div>